# 天気図パターン分類 - 手元で使う (v2)

VS Code でこのノートブックを開いて、上から順に実行してください。
**このリポジトリのフォルダの中だけで完結します**(GitHubにもColabにも繋ぎません)。

Colab版(`notebooks/predict.ipynb`)と**同じ関数**を呼んでいます
(`src/quicklook.py`)。片方だけ直して食い違うことがないよう、中身は1か所に
まとめてあります。

置き場所の決まり:

| | 場所 |
|---|---|
| 2023年以降の天気図(前処理後) | `data/processed/jma/` |
| 2000〜2022年の天気図(前処理後) | `data/processed/ndl/` |
| 重み | `weights/model.pt`・`weights/model_annot.pt` |
| H/L のテンプレート | `data/templates/` |

揃っているかは `python -m scripts.check_local_setup` で確かめられます。
足りないものがあれば、用意するコマンドまで出ます。

In [ ]:
# セットアップ(最初に1回だけ)
import sys
from pathlib import Path

# リポジトリのルートを import できるようにする。VS Code は
# ノートブックのある場所をカレントにすることがあるので、両方に対応する
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline

from src.quicklook import annotation_available, classify_and_show

ok, missing = annotation_available()
print(f"リポジトリ: {ROOT}")
print("注釈方式: " + ("使えます" if ok else f"使えません(足りない: {missing})"))
print()
print("揃っているかまとめて見るには、ターミナルで:")
print("  python -m scripts.check_local_setup")


## 天気図を1枚分類する

In [ ]:
# ここを書き換えて実行する
#
# **パスはすべてこのリポジトリの中で完結させる。**ROOT からの相対で書けば、
# ノートブックをどこから開いても、フォルダごと移動しても動く。
IMAGE = ROOT / "data" / "processed" / "jma" / "Js_2023010100.png"

# 表示するラベルのしきい値。None にすると校正ファイルのラベルごとの値を使う
THRESHOLD = 0.5

# 検出した枠を描き込んでから分類する(左端の絵で検出の当たり外れが見える)
USE_ANNOTATION = True

classify_and_show(IMAGE, threshold=THRESHOLD, annotate=USE_ANNOTATION)


## 古い天気図(2000〜2022年)を渡す場合

In [ ]:
# 2000〜2022年の天気図を渡すとき
#
# テンプレートは2023年以降の天気図から切り出したものなので、そのままでは
# 大きさが3.2%違い、スコアも少し下がる。次の2つを足すと2023年以降と同じ
# 水準で検出できる(README の「検出が0個になるとき」を参照)。
#
#   letter_size="auto"     data/templates/reference.json の基準幅との比で自動調整
#   detect_threshold=0.55  既定の0.65だと取りこぼす
#
# **分類の確信度は当てになりません。**学習に使ったのは2023年以降だけなので、
# 古い天気図はモデルにとって見たことのない絵です。枠が正しく付くかを
# 確かめる用途に使ってください。

OLD_IMAGE = ROOT / "data" / "processed" / "ndl" / "JS_2000010100_page001.png"

classify_and_show(OLD_IMAGE, threshold=0.5, annotate=True,
                  letter_size="auto", detect_threshold=0.55)
